# BSV PDF Downloader

Downloads BSV PDFs from the PHB SPARQL URL list into the shared folder `/projects/agrifutur/bsv_pdfs/`.
Before downloading each PDF, checks if the filename already exists in the folder, skips it if so.
Can be safely interrupted and restarted at any time.

## 1. Configuration

In [1]:
import os, re, time, urllib.request, threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

URL_FILE    = "/projects/agrifutur/yabusaqe/BSV Sample Construction/PDF_URLS.txt"
PDF_DIR = "/projects/agrifutur/yabusaqe/BSV Sample Construction/bsv_pdfs"
WORKERS     = 10
LIMIT       = None  # None = download all, set to e.g. 1000 for a subset
TIMEOUT     = 60    # seconds per PDF
MAX_RETRIES = 2
MIN_SIZE    = 5000  # bytes, skip empty/corrupt PDFs

os.makedirs(PDF_DIR, exist_ok=True)
print(f"PDF folder : {PDF_DIR}")
print(f"Already in folder: {len(os.listdir(PDF_DIR))} files")

PDF folder : /projects/agrifutur/yabusaqe/BSV Sample Construction/bsv_pdfs
Already in folder: 0 files


## 2. Load URLs

In [2]:
with open(URL_FILE, encoding="utf-8", errors="replace") as f:
    content = f.read()

all_urls = list(dict.fromkeys(re.findall(r'https://[^\s<>"]+\.pdf', content)))

if LIMIT:
    all_urls = all_urls[:LIMIT]

print(f"Total unique URLs : {len(all_urls)}")
print(f"Example           : {all_urls[0]}")

Total unique URLs : 20215
Example           : https://rdf.codex.cati.inrae.fr/bsv/files/pdf/Q13917/2019/BSV_HORTI_2019_N15_cle8dbac1.pdf


## 3. Filter, skip already downloaded

In [3]:
shared_folder  = "/projects/agrifutur/bsv_pdfs"
personal_folder = "/projects/agrifutur/yabusaqe/BSV Sample Construction/bsv_pdfs"

existing = set(os.listdir(shared_folder)) | set(os.listdir(personal_folder))

to_download = [
    url for url in all_urls
    if Path(url).name not in existing
]

print(f"In shared folder   : {len(os.listdir(shared_folder))}")
print(f"In personal folder : {len(os.listdir(personal_folder))}")
print(f"Already have       : {len(all_urls) - len(to_download)}")
print(f"Remaining          : {len(to_download)}")

In shared folder   : 1000
In personal folder : 0
Already have       : 1000
Remaining          : 19215


## 4. Download

In [4]:
counts = {"downloaded": 0, "skipped": 0, "errors": 0}
lock   = threading.Lock()

def download_one(url):
    filename = Path(url).name
    dest     = os.path.join(PDF_DIR, filename)

    for attempt in range(MAX_RETRIES + 1):
        try:
            req  = urllib.request.Request(url, headers={"User-Agent": "BSV-Downloader/1.0"})
            with urllib.request.urlopen(req, timeout=TIMEOUT) as resp:
                data = resp.read()
            if len(data) < MIN_SIZE:
                return "skipped"
            with open(dest, "wb") as f:
                f.write(data)
            return "downloaded"
        except Exception:
            if attempt == MAX_RETRIES:
                return "errors"
            time.sleep(2)

with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    futures = {executor.submit(download_one, url): url for url in to_download}
    for i, future in enumerate(as_completed(futures), 1):
        result = future.result()
        with lock:
            counts[result] += 1
        if i % 100 == 0 or i <= 5:
            print(f"[{i}/{len(to_download)}] "
                  f"downloaded={counts['downloaded']} "
                  f"skipped={counts['skipped']} "
                  f"errors={counts['errors']}")

print(f"\nDone. Downloaded={counts['downloaded']} Skipped={counts['skipped']} Errors={counts['errors']}")
print(f"Total PDFs in folder: {len(os.listdir(PDF_DIR))}")

[1/19215] downloaded=1 skipped=0 errors=0
[2/19215] downloaded=2 skipped=0 errors=0
[3/19215] downloaded=3 skipped=0 errors=0
[4/19215] downloaded=4 skipped=0 errors=0
[5/19215] downloaded=5 skipped=0 errors=0
[100/19215] downloaded=100 skipped=0 errors=0
[200/19215] downloaded=200 skipped=0 errors=0
[300/19215] downloaded=300 skipped=0 errors=0
[400/19215] downloaded=400 skipped=0 errors=0
[500/19215] downloaded=500 skipped=0 errors=0
[600/19215] downloaded=600 skipped=0 errors=0
[700/19215] downloaded=700 skipped=0 errors=0
[800/19215] downloaded=800 skipped=0 errors=0
[900/19215] downloaded=900 skipped=0 errors=0
[1000/19215] downloaded=1000 skipped=0 errors=0
[1100/19215] downloaded=1100 skipped=0 errors=0
[1200/19215] downloaded=1200 skipped=0 errors=0
[1300/19215] downloaded=1300 skipped=0 errors=0
[1400/19215] downloaded=1400 skipped=0 errors=0
[1500/19215] downloaded=1500 skipped=0 errors=0
[1600/19215] downloaded=1600 skipped=0 errors=0
[1700/19215] downloaded=1700 skipped=0 e